# Deploy by Command â€” Colab runtime

Re-implementation of *Deploy by Command: A Safe LLM-Driven Platform for Automated AWS EC2 and Docker Deployment from Text Commands*.

This notebook runs the whole project on a **Colab runtime**. The runtime cannot see your local files, so it clones the project from GitHub, installs Node.js 22, and then:

| Step | What it does | Reproduces |
|---|---|---|
| 1 | Configuration (the **only** cell you edit) | â€” |
| 2â€“3 | Node.js 22, clone, `npm ci` | â€” |
| 4 | Run the 29 automated tests | Table III |
| 5 | Rule-classifier evaluation | Table IV |
| 6 | Live Gemini observation (only if a key is set) | Table V |
| 7â€“8 | Build the dashboard, start the backend in **MODE=mock** | Fig. 1, Fig. 9/10 |
| 9 | Simulated end-to-end run from Python | Sec. 4 workflow |
| 10 | Benchmark + plot | Table VIII, Fig. 14 |
| 11 | Open the dashboard (Colab port proxy or Cloudflare quick tunnel) | Fig. 9/10 |
| 12 | Optional real AWS (disabled by default) | Module 4/5 |
| 13 | Cleanup | â€” |

**How to run:** in VS Code with the Google Colab extension, select a Colab runtime as the kernel, edit cell 1 if needed, then *Run All*.

## 1. Configuration â€” the only cell you edit

* `REPO_URL` â€” your GitHub repository.
* `PRIVATE_REPO` â€” if `True`, you are asked for a GitHub token (read access) with `getpass`. It is never printed.
* The Gemini key is read from **Colab Secrets** (`GEMINI_API_KEY`, enable notebook access in the ðŸ”‘ panel). If it is not there you are asked with `getpass`; press Enter to skip and use the rule-based engine only.
* `RUN_REAL_AWS` / `I_ACCEPT_BILLING` â€” the optional real-AWS section (step 12). Leave them `False` unless you mean it.

In [ ]:
REPO_URL = "https://github.com/Saipavanavsp/deploy-by-command.git"
PRIVATE_REPO = True        # True -> asks for a GitHub token (getpass) to clone a private repo
RUN_REAL_AWS = False         # step 12: real AWS pre-flight (creates nothing)
I_ACCEPT_BILLING = False     # step 12: also launch + deploy on real AWS (billable!)

import os, getpass, secrets

def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

GITHUB_TOKEN = getpass.getpass("GitHub token for the private repo: ").strip() if PRIVATE_REPO else ""
GEMINI_API_KEY = colab_secret("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Gemini API key (Enter to skip -> rules engine only): ").strip()

PROJECT_DIR = "/content/deploy-by-command"
BACKEND_DIR = f"{PROJECT_DIR}/backend"
ACCESS_TOKEN = secrets.token_urlsafe(24)   # protects the dashboard when it is tunnelled (step 11)
print("Repository :", REPO_URL)
print("Gemini key :", "provided (hidden)" if GEMINI_API_KEY else "not set -> rule-based engine only")
print("Real AWS   :", "pre-flight enabled" if RUN_REAL_AWS else "disabled", "| billing accepted" if I_ACCEPT_BILLING else "")

## 2. Install Node.js 22 (NodeSource)

The backend, tests and dashboard build all need Node.js 22.

In [ ]:
%%bash
set -e
if ! node -v 2>/dev/null | grep -q '^v22'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /tmp/nodesource.log 2>&1
  apt-get install -y nodejs > /tmp/node-install.log 2>&1
fi
echo "node $(node -v)  npm $(npm -v)"

## 3. Clone the repository and install dependencies

Clones into `/content/deploy-by-command`, then runs `npm ci` in `backend/` and `frontend/`. A token, if used, is removed from the output and from the saved git remote.

In [ ]:
import shutil, subprocess

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
r = subprocess.run(["git", "clone", "--depth", "1", url, PROJECT_DIR], capture_output=True, text=True)
out = r.stdout + r.stderr
if GITHUB_TOKEN:
    out = out.replace(GITHUB_TOKEN, "[REDACTED]")
print(out.strip())
assert r.returncode == 0, "git clone failed (check REPO_URL / token)"
subprocess.run(["git", "-C", PROJECT_DIR, "remote", "set-url", "origin", REPO_URL])
print("Cloned to", PROJECT_DIR)

In [ ]:
%%bash
cd /content/deploy-by-command/backend && npm ci --no-audit --no-fund --loglevel=error
cd /content/deploy-by-command/frontend && npm ci --no-audit --no-fund --loglevel=error
echo "dependencies installed"

## 4. Run the 29 automated tests â€” Table III

`node:test` + `supertest` against the real HTTP interface, using the mock provider and a stubbed Gemini `fetch`. Expect **29/29**.

In [ ]:
%%bash
cd /content/deploy-by-command/backend && npm test 2>&1 | tail -n 45

## 5. Rule-classifier evaluation â€” Table IV

The 12 reference commands; expect **12 / 12**. Also shows that the free-form sentence *"can you put my site on the internet please"* is **not** understood by the rules (UNKNOWN, confidence 0.2, nothing runs).

In [ ]:
%%bash
cd /content/deploy-by-command/backend && node scripts/eval-classifier.js

## 6. Live Gemini observation â€” Table V (only if a key is set)

Classifies *"Deploy my website"* twice and the free-form sentence once, through Gemini's OpenAI-compatible endpoint. Only the command text is sent. The key is passed to the subprocess only and redacted from any output.

In [ ]:
if GEMINI_API_KEY:
    env = {**os.environ, "GEMINI_API_KEY": GEMINI_API_KEY}
    r = subprocess.run(["node", "scripts/observe-gemini.js"], cwd=BACKEND_DIR, env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(GEMINI_API_KEY, "[REDACTED]"))
else:
    print("Table V â€” not measured: no Gemini key was provided, so Gemini was not called.")

## 7. Build the dashboard (`vite build`)

The backend serves `frontend/dist` as static files, so the dashboard and the API share one origin (port 4000).

In [ ]:
%%bash
cd /content/deploy-by-command/frontend && npm run build 2>&1 | tail -n 8

## 8. Start the backend in the background â€” MODE=mock

`MODE=mock` (simulated: nothing touches AWS, SSH or Docker). `COLAB_TUNNEL=1` makes every request require the random `ACCESS_TOKEN` generated in step 1, because the dashboard has no login and may be tunnelled in step 11. The backend still binds `127.0.0.1`.

This cell also defines small Python helpers (`api`, `wait_idle`, `follow`) used below.

In [ ]:
import json, time, requests

BACKEND = "http://127.0.0.1:4000"
HEADERS = {"x-access-token": ACCESS_TOKEN}
procs = {}

def stop_backend():
    p = procs.pop("backend", None)
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(10)
        except Exception:
            p.kill()
    subprocess.run("pkill -f 'node src/server.js'", shell=True)

def start_backend(extra_env):
    stop_backend()
    env = {**os.environ, "PORT": "4000", "HOST": "127.0.0.1", "ENV_FILE": "/content/.no-env-file",
           "GEMINI_API_KEY": GEMINI_API_KEY, **extra_env}
    log = open("/content/backend.log", "w")
    procs["backend"] = subprocess.Popen(["nohup", "node", "src/server.js"], cwd=BACKEND_DIR, env=env,
                                        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        try:
            if requests.get(f"{BACKEND}/health", headers=HEADERS, timeout=2).ok:
                break
        except requests.RequestException:
            pass
        if procs["backend"].poll() is not None:
            break
        time.sleep(0.5)
    print(open("/content/backend.log").read())

def api(method, path, body=None):
    r = requests.request(method, f"{BACKEND}/api{path}", headers=HEADERS, json=body, timeout=30)
    return r.status_code, r.json()

def backend_mode():
    try:
        return api("GET", "/health")[1].get("mode")
    except Exception:
        return None

last_log = {"id": 0}
def follow():
    _, d = api("GET", f"/logs?since={last_log['id']}")
    for l in d["logs"]:
        print(f"{l['ts'][11:19]} {l['level']:<5} {l['msg']}")
    last_log["id"] = d["lastId"]

def wait_idle(poll=0.5, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        follow()
        s = api("GET", "/state")[1]
        if not s["busy"]:
            follow()
            return s
        time.sleep(poll)
    raise TimeoutError("operation still running")

# fresh simulated state for a deterministic demo
mock_state = f"{BACKEND_DIR}/state/state-mock.json"
if os.path.exists(mock_state):
    os.remove(mock_state)
start_backend({"MODE": "mock", "ALLOW_REAL_AWS": "no", "COLAB_TUNNEL": "1", "ACCESS_TOKEN": ACCESS_TOKEN, "MOCK_DELAY_MS": "700"})
print("health:", api("GET", "/health")[1])

## 9. Simulated end-to-end run from Python

1. `POST /command` *"Deploy my website"* â†’ interpretation only (runs nothing).
2. `POST /full` â†’ **202 Accepted**; the work runs in the background. We follow `/logs` and `/state` until done. Every provider line is tagged **[SIMULATED]**, and the IP is from the reserved documentation range 198.51.100.0/24.

In [ ]:
code_, interp = api("POST", "/command", {"text": "Deploy my website"})
print(json.dumps(interp, indent=2))

code_, accepted = api("POST", f"/{interp['operation']}", {})
print("\nHTTP", code_, accepted, "\n")
state = wait_idle()
print("\nSteps :", state["steps"])
print("Result:", state["lastResult"]["message"])
print("Server:", state["infra"]["instanceId"], "| IP", state["infra"]["publicIp"], "(example address)")

Run it again: finished steps are **skipped** ("skip launch", "skip configure") and only the deploy step runs. A second launch is refused while a server is tracked.

In [ ]:
code_, accepted = api("POST", "/full", {})
print("HTTP", code_, accepted, "\n")
state = wait_idle()
print("\nResult:", state["lastResult"]["message"])
print("Second launch ->", api("POST", "/launch", {}))

A destructive or unclear command maps to **UNKNOWN**: no action, nothing runs. Terminate is never reachable from command text.

In [ ]:
for text in ["delete everything", "terminate my instance"]:
    code_, r = api("POST", "/command", {"text": text})
    print(f'"{text}" -> {r["intent"]} (engine {r["engine"]}, confidence {r["confidence"]}), operation={r["operation"]}')
    print("   reason:", r["reason"])
s = api("GET", "/state")[1]
print("\nStill one tracked server:", s["infra"]["instanceId"], "| busy:", s["busy"])

## 10. Benchmark â€” Table VIII, and plot â€” Fig. 14

50 sequential requests each to `GET /health`, `POST /command` (rules) and `GET /logs` on loopback in mock mode (median and 95th percentile), a burst of 260 requests to show the **240/min â†’ 429** limit, the test count/time, and the JS/CSS bundle sizes from `vite build`. Anything not measured is labelled *not measured*.

In [ ]:
%%bash
cd /content/deploy-by-command/backend && node scripts/benchmark.js

In [ ]:
from IPython.display import Image, display
r = subprocess.run(["python3", "scripts/plot.py"], cwd=BACKEND_DIR, capture_output=True, text=True)
print(r.stdout + r.stderr)
if os.path.exists(f"{BACKEND_DIR}/results/fig14.png"):
    display(Image(f"{BACKEND_DIR}/results/fig14.png"))

## 11. Open the dashboard

First try Colab's port proxy. Through VS Code this may not open; if so, use the **Cloudflare quick tunnel** in the next cell.

âš ï¸ The dashboard has no login. In this notebook it stays in **MODE=mock** and every request needs the random access token from step 1 (it is part of the printed link). Do not share the link. Stop the tunnel with the cell after it.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(4000, path=f"/?token={ACCESS_TOKEN}", anchor_text="Open the Deploy by Command dashboard")
except Exception as e:
    print("Colab port proxy is not available here:", e)
    print("Use the Cloudflare quick tunnel below.")

In [ ]:
import re
CLOUDFLARED = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED):
    subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod(CLOUDFLARED, 0o755)
if procs.get("tunnel") and procs["tunnel"].poll() is None:
    procs["tunnel"].terminate()
tlog = open("/content/tunnel.log", "w")
procs["tunnel"] = subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:4000"],
                                   stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
tunnel_url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        tunnel_url = m.group(0)
        break
    time.sleep(1)
if tunnel_url:
    print("WARNING: anyone with this full link can use the (simulated) dashboard until you stop the tunnel.")
    print("Dashboard:", f"{tunnel_url}/?token={ACCESS_TOKEN}")
    print("(It may take ~10-30 s before the new hostname resolves.)")
else:
    print("Tunnel did not start; see /content/tunnel.log")

### Stop the tunnel

In [ ]:
p = procs.pop("tunnel", None)
if p and p.poll() is None:
    p.terminate()
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
print("Tunnel stopped.")

## 12. Optional: real AWS (disabled by default)

**Read before enabling.** Real AWS needs two separate locks: `MODE=aws` **and** `ALLOW_REAL_AWS=yes` (this cell sets both only when `RUN_REAL_AWS = True`).

* **Cost:** a `t3.micro` in `ap-south-1` plus a public IPv4 address is billable (small, but not zero) until you terminate it.
* **Credentials:** add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` of the dedicated least-privilege IAM user (see README) to Colab Secrets. They are passed to the backend process only.
* **SSH CIDR:** port 22 is opened to **this Colab VM's public IP /32**. That IP changes between sessions, so a later session cannot SSH into the server (Terminate still works, it only uses the AWS API).
* With only `RUN_REAL_AWS = True`, this runs the **pre-flight checks only** (identity, instance type, AMI, default VPC, dry run). Nothing is created.
* With `I_ACCEPT_BILLING = True` as well, it launches, configures and deploys. The tunnel is stopped first (tunnelling is only allowed in mock mode). **Always run the Terminate cell after.**

In [ ]:
def aws_env():
    return {"MODE": "aws", "ALLOW_REAL_AWS": "yes", "AWS_REGION": "ap-south-1", "INSTANCE_TYPE": "t3.micro",
            "AWS_ACCESS_KEY_ID": colab_secret("AWS_ACCESS_KEY_ID"),
            "AWS_SECRET_ACCESS_KEY": colab_secret("AWS_SECRET_ACCESS_KEY"),
            "AWS_PROFILE": "", "COLAB_TUNNEL": "0", "ENV_FILE": "/content/.no-env-file"}

def redact_aws(text):
    for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
        v = colab_secret(k)
        if v:
            text = text.replace(v, "[REDACTED]")
    return text

if not RUN_REAL_AWS:
    print("Skipped: RUN_REAL_AWS = False. Nothing touched AWS.")
elif not (colab_secret("AWS_ACCESS_KEY_ID") and colab_secret("AWS_SECRET_ACCESS_KEY")):
    print("Skipped: add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY to Colab Secrets first.")
else:
    r = subprocess.run(["node", "scripts/preflight.js"], cwd=BACKEND_DIR, env={**os.environ, **aws_env()},
                       capture_output=True, text=True)
    print(redact_aws(r.stdout + r.stderr))
    if r.returncode != 0:
        print("Pre-flight failed: nothing was created.")
    elif not I_ACCEPT_BILLING:
        print("Pre-flight only. Set I_ACCEPT_BILLING = True (step 1) and re-run to launch billable resources.")
    else:
        p = procs.pop("tunnel", None)
        if p and p.poll() is None:
            p.terminate()
        start_backend(aws_env())
        last_log["id"] = 0
        print(api("POST", "/full", {}))
        state = wait_idle(poll=3)
        print("\nResult:", state["lastResult"]["message"])
        if state.get("deployment"):
            print("Website:", state["deployment"]["url"])

### Terminate the real server (always run this after a real launch)

Sends `POST /terminate` with the typed confirmation `"TERMINATE"`, then waits: instance terminated â†’ security group deleted â†’ key pair deleted â†’ local key file deleted â†’ state cleared.

In [ ]:
aws_state_file = f"{BACKEND_DIR}/state/state-aws.json"
tracked = None
if os.path.exists(aws_state_file):
    tracked = (json.load(open(aws_state_file)).get("infra") or {}).get("instanceId")
if not tracked:
    print("No real instance is tracked. Nothing to terminate.")
else:
    if backend_mode() != "aws":
        start_backend(aws_env())
    print(api("POST", "/terminate", {"confirm": "TERMINATE"}))
    state = wait_idle(poll=3)
    print("\nResult:", state["lastResult"]["message"])

## 13. Cleanup

Terminates a real instance if one is still tracked, then stops the backend and the tunnel.

In [ ]:
aws_state_file = f"{BACKEND_DIR}/state/state-aws.json"
tracked = None
if os.path.exists(aws_state_file):
    tracked = (json.load(open(aws_state_file)).get("infra") or {}).get("instanceId")
if tracked:
    print(f"Real instance {tracked} is still tracked: terminating it.")
    try:
        if backend_mode() != "aws":
            start_backend(aws_env())
        print(api("POST", "/terminate", {"confirm": "TERMINATE"}))
        print(wait_idle(poll=3)["lastResult"]["message"])
    except Exception as e:
        print("Could not terminate automatically:", e)
        print(f"Terminate {tracked} in the AWS console (ap-south-1), then delete its security group and key pair.")
for name in ("tunnel",):
    p = procs.pop(name, None)
    if p and p.poll() is None:
        p.terminate()
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
stop_backend()
print("Backend and tunnel stopped.")